In [25]:
import pandas as pd
pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv('data/processed/dataset_sucio.csv')

In [3]:
# limpiamos los nulos, duplicados y columnas innecesarias

In [4]:
eliminar = []

for col in df.columns:
    if df[col].isna().sum() > len(df) * 0.15:
        eliminar.append(col)

print(eliminar)


['pobr', 'fum', 'alc', 'obes', 'fyv', 'sati']


In [5]:
df = df.drop(columns= eliminar)

Las variables 'crim' y 'tasa_criminalidad' son iguales... en 'crim' tenemos mas nulos por que no tenemos datos de Castilla Y León, esto seguramente se debe a una perdida de datos a la hora de formar el dataset.  
Las varibles 'precio_medio_anual_eur_m2_venta','precio_medio_anual_eur_m2_alquiler' faltan datos de algunos años de Ceuta, Melilla, Navarra, País Vasco y Rioja. No parece ser una perdida de datos si no a la no existencia de los mismos.

In [6]:
df = df.drop(columns= ['crim'])

In [7]:
col_con_nulos = ['homi','tasa_criminalidad', 'precio_medio_anual_eur_m2_venta','precio_medio_anual_eur_m2_alquiler'] 

In [8]:
for col in col_con_nulos:
    bfill = df.groupby("com_aut")[col].bfill()
    mask = (df["año"] == 2009) & (df[col].isna())
    df.loc[mask, col] = bfill[mask]

In [9]:
cols_precios = ["precio_medio_anual_eur_m2_venta", "precio_medio_anual_eur_m2_alquiler"]
df = df.sort_values(["com_aut", "año"]).copy()

df[cols_precios] = (
    df.groupby("com_aut")[cols_precios]
      .transform(lambda x: x.interpolate(method="linear", limit_area="inside"))
)

In [10]:
df = df.sort_values(["com_aut", "año"]).copy()

df[cols_precios] = df.groupby("com_aut")[cols_precios].ffill()

In [11]:
df[df.duplicated()]

,año,com_aut,pib,pob,sui,nat,paro,ing,homi,pib_pc,...,retrasos_pagos(%),renta_media,renta_mediana,riesgo_pobreza(%),dificultad_fin_mes(%),desigualdad_ing(S80/S20),inc_gastos_imprevistos(%),tasa_criminalidad,precio_medio_anual_eur_m2_venta,precio_medio_anual_eur_m2_alquiler


In [13]:
df = df.drop(columns= ['pob','pib_pc'])

In [16]:
df['pib_pc']= df['pib']/df['poblacion']

Index(['año', 'com_aut', 'pib', 'sui', 'nat', 'paro', 'ing', 'homi',
       'renta_pc', 'poblacion', 'gasto_elevado_vivienda(%)',
       'falta_espacio_vivienda(%)', 'retrasos_pagos(%)', 'renta_media',
       'renta_mediana', 'riesgo_pobreza(%)', 'dificultad_fin_mes(%)',
       'desigualdad_ing(S80/S20)', 'inc_gastos_imprevistos(%)',
       'tasa_criminalidad', 'precio_medio_anual_eur_m2_venta',
       'precio_medio_anual_eur_m2_alquiler', 'pib_pc'],
      dtype='object')

In [19]:
df = df.rename(columns = {'com_aut':'comunidad_autonoma',
                     'pib':'pib_total',
                     'sui':'tasa_suicidio',
                     'nat':'tasa_natalidad',
                     'ing':'ingresos_pc',
                     'homi':'tasa_homicidio',
                     'precio_medio_anual_eur_m2_venta': 'precio_venta_m2',
                     'precio_medio_anual_eur_m2_alquiler': 'precio_alquiler_m2',
                          'falta_espacio_vivienda(%)':'falta_espacio_vivienda',
                          'gasto_elevado_vivienda(%)':'gasto_elevado_vivienda',
                          'dificultad_fin_mes(%)':'dificultad_fin_mes',
                          'inc_gastos_imprevistos(%)':'inc_gastos_imprevistos',
                          'retrasos_pagos(%)':'retrasos_pagos',
                          'riesgo_pobreza(%)':'riesgo_pobreza'
                          
    
})

In [24]:
df = df[['año', 'comunidad_autonoma','poblacion', 'pib_total','pib_pc', 'ingresos_pc',
    'renta_pc',
    'renta_media', 'renta_mediana','tasa_natalidad', 'tasa_suicidio', 'tasa_homicidio', 
    'tasa_criminalidad','paro','gasto_elevado_vivienda', 'falta_espacio_vivienda',
    'retrasos_pagos','riesgo_pobreza', 'dificultad_fin_mes','desigualdad_ing(S80/S20)',
    'inc_gastos_imprevistos','precio_venta_m2', 'precio_alquiler_m2'
       ]]

df

,año,comunidad_autonoma,poblacion,pib_total,pib_pc,ingresos_pc,renta_pc,renta_media,renta_mediana,tasa_natalidad,tasa_suicidio,tasa_homicidio,tasa_criminalidad,paro,gasto_elevado_vivienda,falta_espacio_vivienda,retrasos_pagos,riesgo_pobreza,dificultad_fin_mes,desigualdad_ing(S80/S20),inc_gastos_imprevistos,precio_venta_m2,precio_alquiler_m2
0,2009,Andalucía,7627743.0,145802256.0,19.114731,9437.0,11869.0,143930.0,126950.0,11.48,10.87,0.94,49.3,17.73,8.6,5.7,10.6,27.7,71.8,6.0,45.7,1757.50,7.27
1,2010,Andalucía,7666919.0,145853901.0,19.023796,9882.0,11773.0,141520.0,120860.0,11.10,9.89,0.94,49.3,25.24,7.8,4.6,12.9,28.1,73.9,6.2,52.2,1713.33,6.85
2,2011,Andalucía,7693947.0,144494494.0,18.780282,9793.0,11797.0,13310.0,113690.0,10.72,8.84,1.02,48.7,27.77,11.2,7.5,13.8,34.5,70.5,7.3,49.6,1723.42,6.72
3,2012,Andalucía,7702875.0,139052618.0,18.052041,9241.0,11155.0,133910.0,111170.0,10.30,8.30,0.81,47.1,30.13,8.7,6.6,15.5,35.6,71.6,6.2,51.4,1579.42,6.43
4,2013,Andalucía,7710575.0,137294091.0,17.805947,9361.0,11100.0,127050.0,108660.0,9.71,9.78,0.78,44.6,34.35,9.2,6.4,13.4,40.1,75.4,6.6,55.5,1443.42,6.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,2018,Rioja,280018.0,8649852.0,30.890343,12561.0,16401.0,17490.0,15410.0,7.45,7.62,0.32,26.3,12.00,9.4,0.0,7.9,19.1,37.8,5.7,30.3,1180.17,5.89
262,2019,Rioja,279568.0,8889997.0,31.799051,12589.0,16765.0,186610.0,165770.0,7.66,6.11,0.00,27.6,10.40,5.8,2.1,4.9,17.5,25.4,5.0,19.0,1205.67,6.21
263,2020,Rioja,279704.0,8186915.0,29.269925,13371.0,16489.0,196980.0,175890.0,7.33,8.75,1.57,24.5,9.96,4.6,2.6,8.2,16.9,27.7,4.7,24.5,1215.08,6.58
264,2021,Rioja,278905.0,8735750.0,31.321597,14303.0,16951.0,187580.0,17420.0,6.94,8.27,0.94,28.6,10.80,8.2,3.7,8.0,16.8,32.3,5.1,22.5,1238.25,6.83


In [22]:
df.to_csv("data/processed/dataset_final.csv", index=False)

### Resumen del proceso de limpieza del dataset

En primer lugar, se realizó un análisis de valores ausentes por columna. A partir de este análisis, se decidió eliminar aquellas columnas que presentaban más del 15 % de valores nulos respecto al total de filas del dataset. Para ello, se calculó el umbral multiplicando el número total de filas por 0,15, y se eliminaron todas las columnas cuyo número de valores ausentes superaba dicho umbral.

En una segunda etapa, se identificaron variables redundantes que representaban la misma información. En estos casos, se conservó una única variable y se eliminó la columna duplicada para evitar introducir ruido o duplicación innecesaria en el análisis.

Posteriormente, se abordó el tratamiento de variables que presentaban valores ausentes concentrados principalmente en el año inicial del dataset (2009). Dado que el conjunto de datos comienza en ese año y que para algunas comunidades no existían datos oficiales en 2009 pero sí en 2010, se optó por imputar dichos valores utilizando el dato correspondiente al año 2010 dentro de la misma comunidad autónoma, preservando así la continuidad temporal sin eliminar el año completo.

Por último, para las variables de precio medio anual de vivienda (venta y alquiler), se aplicó una estrategia de imputación en dos pasos. En primer lugar, se realizó una interpolación temporal lineal intra-comunidad para completar huecos intermedios entre años con datos disponibles. En segundo lugar, para los valores ausentes que permanecían en los años finales de la serie (al no existir un año posterior que permitiera la interpolación), se utilizó un forward fill intra-comunidad, imputando dichos valores con el último dato disponible para la misma comunidad. Este procedimiento se limitó exclusivamente a las variables de precios y se consideró una aproximación conservadora.



| Variable | Descripción | Tipo de variable | Importancia inicial |
|---------|-------------|------------------|---------------------|
| año | Año de referencia de los datos. | Numérica discreta | 0 |
| comunidad_autonoma | Comunidad Autónoma de España. | Categórica nominal | 0 |
| poblacion | Número total de habitantes por comunidad y año. | Numérica discreta | 3 |
| pbi_total | Producto Interior Bruto total. Valor económico agregado. | Numérica continua | 0 |
| pib_pc | PIB per cápita a precios de mercado. | Numérica continua | 1 |
| ingresos | Ingresos medios según la fuente del dataset. | Numérica continua | 1 |
| renta_pc | Renta disponible bruta de los hogares per cápita. | Numérica continua | 1 |
| renta_media | Renta media por unidad de consumo. | Numérica continua | 1 |
| renta_mediana | Renta mediana por unidad de consumo. | Numérica continua | 1 |
| tasa_natalidad | Nacimientos por cada 1.000 habitantes. | Numérica continua | 2 |
| tasa_suicidio | Defunciones por suicidio por 100.000 habitantes. | Numérica continua | 2 |
| tasa_homicidio | Homicidios y asesinatos por 100.000 habitantes. | Numérica continua | 3 |
| tasa_criminalidad | Delitos o infracciones penales por 1.000 habitantes. | Numérica continua | 2 |
| paro | Tasa de desempleo sobre población activa. | Numérica continua | 1 |
| gasto_ele_vivienda(%) | Población con gasto elevado en vivienda. | Numérica continua | 2 |
| falta_espacio_vivienda(%) | Población que vive en viviendas con falta de espacio. | Numérica continua | 3 |
| retrasos_pagos(%) | Población con retrasos en pagos de vivienda o suministros. | Numérica continua | 2 |
| riesgo_pobreza(%) | Población en riesgo de pobreza. | Numérica continua | 1 |
| dificultad_fin_mes(%) | Población con dificultad para llegar a fin de mes. | Numérica continua | 2 |
| desigualdad_ing(S80/S20) | Cociente entre el 20% más rico y el 20% más pobre. | Numérica continua | 1 |
| inc_gastos_imprevistos(%) | Población que no puede afrontar gastos imprevistos. | Numérica continua | 2 |
| precio_alq_eur_m2 | Precio medio anual de vivienda en alquiler por m² (€). | Numérica continua | 2 |
| precio_venta_eur_m2 | Precio medio anual de vivienda en venta por m² (€). | Numérica continua | 2 |
